In [1]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.resnet50 import create_resnet50
from src.data.dataloader import create_dataloaders

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.0+cu132
CUDA available: True


Create ResNet50

In [3]:
model = create_resnet50(
    num_classes=10,
    pretrained=True,
    freeze_backbone=True,
)

model = model.to(device)

print(model.fc)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\USER/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:30<00:00, 3.40MB/s]


Linear(in_features=2048, out_features=10, bias=True)


Check trainable parameters

In [4]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}")

Total parameters:     23,528,522
Trainable parameters: 20,490
Frozen parameters:    23,508,032


In [5]:
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name, parameter.shape)

fc.weight torch.Size([10, 2048])
fc.bias torch.Size([10])


Load the shared data pipeline

In [6]:
loaders = create_dataloaders(
    dataset_root=PROJECT_ROOT / "data/raw/garbage-dataset",
    manifest_dir=PROJECT_ROOT / "data/manifests",
    batch_size=32,
    num_workers=0,
)

train_loader = loaders["train_loader"]
val_loader = loaders["val_loader"]

print("Train:", len(loaders["train_dataset"]))
print("Validation:", len(loaders["val_dataset"]))
print("Classes:", loaders["class_to_idx"])

Train: 13833
Validation: 2964
Classes: {'battery': 0, 'biological': 1, 'cardboard': 2, 'clothes': 3, 'glass': 4, 'metal': 5, 'paper': 6, 'plastic': 7, 'shoes': 8, 'trash': 9}


Test one batch through ResNet50

In [7]:
images, labels = next(iter(train_loader))

print("Input:", images.shape)
print("Labels:", labels.shape)

images = images.to(device)
labels = labels.to(device)

model.eval()

with torch.inference_mode():
    outputs = model(images)

print("Output:", outputs.shape)

Input: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Output: torch.Size([32, 10])


Verify loss calculation

In [8]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

loss = criterion(
    outputs,
    labels,
)

print("Loss:", loss.item())

Loss: 2.332965850830078


Check GPU memory

In [9]:
if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        / 1024**3
    )

    print(
        f"Allocated GPU memory: {allocated:.2f} GB"
    )

    print(
        f"Reserved GPU memory: {reserved:.2f} GB"
    )

Allocated GPU memory: 0.14 GB
Reserved GPU memory: 0.56 GB
